In [43]:
import json 
from pathlib import Path
import pandas as pd 
import joblib 
import numpy as np
import tqdm
import re

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report , accuracy_score

In [26]:
# Specigying model sirectory and data dir
Model_dir = Path('../models')
data = Path('../data/transactions_v2.csv')
clean_data = Path('../data/train_clean.csv')

In [27]:
df = pd.read_csv(data)
df2 = pd.read_csv(clean_data)

In [18]:
df.head()

,date,description,amount,label,source
0,2023-08-17,Card Rent Pmt Rent,985.04,housing,simulated_bank_transactions_uk
1,2023-03-31,pos/direct/debit/vodafone,226.28,utilities,simulated_bank_transactions_uk
2,2023-08-27,Subscription/Gym/Group 357778,8.28,subscriptions,simulated_bank_transactions_uk
3,2024-10-27,mastercard-direct-debit-amazon-prime,11.96,subscriptions,simulated_bank_transactions_uk
4,2023-01-07,Mortgage Payment Rent,1424.97,housing,simulated_bank_transactions_uk


In [21]:
df['label'].unique()

array(['housing', 'utilities', 'subscriptions', 'other', 'transportation',
       'groceries', 'dining'], dtype=object)

In [22]:
df['label'].value_counts()

label
utilities         1480
housing           1447
subscriptions     1437
groceries         1436
transportation    1422
dining            1419
other             1359
Name: count, dtype: int64

In [25]:
### Performing Regex on the data 
df["description"] = (
    df["description"]
    .str.lower()
    .str.replace(r"\b\d{5,}\b", "<ref>", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df.to_csv("../data/train_clean.csv", index=False)


In [28]:
df2.head()

,date,description,amount,label,source
0,2023-08-17,card rent pmt rent,985.04,housing,simulated_bank_transactions_uk
1,2023-03-31,pos/direct/debit/vodafone,226.28,utilities,simulated_bank_transactions_uk
2,2023-08-27,subscription/gym/group <ref>,8.28,subscriptions,simulated_bank_transactions_uk
3,2024-10-27,mastercard-direct-debit-amazon-prime,11.96,subscriptions,simulated_bank_transactions_uk
4,2023-01-07,mortgage payment rent,1424.97,housing,simulated_bank_transactions_uk


In [29]:
X = df2["description"]
y = df2["label"]

In [30]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2 , random_state=42 , stratify= y)

In [31]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=3,
        max_features=20000
    )),
    ("clf",LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

In [32]:
model.fit(X_train,y_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [33]:
preds = model.predict(X_test)

In [34]:
print(classification_report(y_test,preds))

                precision    recall  f1-score   support

        dining       1.00      1.00      1.00       284
     groceries       1.00      1.00      1.00       287
       housing       1.00      1.00      1.00       289
         other       1.00      1.00      1.00       272
 subscriptions       1.00      1.00      1.00       287
transportation       1.00      1.00      1.00       285
     utilities       1.00      1.00      1.00       296

      accuracy                           1.00      2000
     macro avg       1.00      1.00      1.00      2000
  weighted avg       1.00      1.00      1.00      2000



### An accuracy of 1 in every class is quite suspicious also considering we used synthetic data in an orderly manner , I will try to evaluate the models in other ways 

#### Test 1 - Shuffling the Labels to test for data leakage 

In [42]:
y_shuffled = np.random.permutation(y_train)

model.fit(X_train, y_shuffled)
preds = model.predict(X_test)

print("Accuracy with shuffled labels:",
      accuracy_score(y_test, preds))

Accuracy with shuffled labels: 0.127


In [45]:
def mask_merchants(text):
    return re.sub(
        r"\b(tesco|morrisons|sainsburys|netflix|spotify|tfl|uber|british gas)\b",
        "<merchant>",
        text
    )

X_masked = X_train.apply(mask_merchants)
X_test_masked = X_test.apply(mask_merchants)

model.fit(X_masked, y_train)
preds = model.predict(X_test_masked)

print(classification_report(y_test, preds))

                precision    recall  f1-score   support

        dining       1.00      1.00      1.00       284
     groceries       1.00      1.00      1.00       287
       housing       1.00      1.00      1.00       289
         other       1.00      1.00      1.00       272
 subscriptions       0.98      1.00      0.99       287
transportation       1.00      1.00      1.00       285
     utilities       1.00      0.98      0.99       296

      accuracy                           1.00      2000
     macro avg       1.00      1.00      1.00      2000
  weighted avg       1.00      1.00      1.00      2000



##### Initial evaluation achieved near-perfect accuracy on synthetic data. To validate this result, we ran a label-shuffling test (accuracy ≈ random chance), confirming the absence of data leakage. We also masked merchant identifiers and observed only a small drop in performance, indicating the model learned structural transaction patterns (e.g. payment rails, recurring charges). The remaining optimism reflects the controlled nature of the synthetic data and would likely decrease on real bank feeds.

In [52]:
joblib.dump(model, "../models/model.joblib")

meta = {
    "model": "tfidf + logistic_regression",
    "labels": sorted(y.unique().tolist()),
}
(Path("../models/metadata.json")).write_text(json.dumps(meta, indent=2))

181